# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Get metadata
meta_json = dataset.metadata.to_json()

print(f"Dataset: {meta_json['name']}")
print(f"Description: {meta_json['description']}")
print(f"Version: {meta_json.get('version', 'N/A')}")
print(f"Published date: {meta_json.get('datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs via Croissant schema.

In [ ]:
# List all record sets and their @id
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"Record Set name: {rs.name}, @id: {rs.id_}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field name: {field.name}, @id: {field.id_}, data_type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Get all record set @ids
record_set_ids = [rs.id_ for rs in dataset.metadata.record_sets]

print(f"Record set @ids: {record_set_ids}")

# We load each record set as a DataFrame
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Loaded {len(df)} records for record set @id: {rsid}")

# For demonstration, use the first record set
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Main record set selected: {main_record_set_id}")
    print("Columns available:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field from the primary record set for demonstration. Make sure to pick the appropriate field `@id` for numeric analysis.

In [ ]:
# Identify available fields in the chosen record set
fields = {field.name: field for field in dataset.metadata["record_sets"][0].fields}
print("Fields in selected record set:")
for k, v in fields.items():
    print(f"- {k} (@id={v.id_}, type={v.data_type})")

# Suppose 'MW' (Molecular Weight) is present and is a float; adjust '@id' accordingly or pick another numeric field you see above
# We'll attempt to use a likely numeric field for demonstration (update field_id if needed)
import numpy as np

numeric_field_id = None
for field in dataset.metadata["record_sets"][0].fields:
    if field.data_type.lower() in ["float", "number", "integer"]:
        numeric_field_id = field.id_
        break

print(f"Selected numeric field: {numeric_field_id}")

df = dataframes[main_record_set_id]

# Try to ensure numeric type
if numeric_field_id is not None and numeric_field_id in df.columns:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()  # Set an example threshold at the mean
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to find a 'group' field (string/categorical)
    group_field_id = None
    for field in dataset.metadata["record_sets"][0].fields:
        if field.data_type.lower() in ["text", "string"] and field.id_ in df.columns:
            group_field_id = field.id_
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id, dropna=False).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No categorical group field found for grouping.")
else:
    print("No numeric field found in the main record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# If both numeric and group_field_id exist, show boxplot
if numeric_field_id and group_field_id and numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(12,6))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we loaded and explored the 'Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells' dataset using the `mlcroissant` library. We examined the record sets and their schema, loaded the main data table, analyzed a representative numeric field, and created summary statistics and visualizations. This dataset enables protein-level quantitative and comparative analyses, useful for downstream bioinformatics workflows or further FAIR dataset benchmarking.*